# धडा १६ - Microsoft Foundry सह स्केलेबल एजंट्सची तैनाती

या नोटबुकमध्ये तुम्ही काल्पनिक कंपनी **Contoso** साठी एक **उत्पादनासाठी तयार ग्राहक समर्थन एजंट** तयार करता. आधीच्या धड्यांपासून वेगळे म्हणजे, येथे एजंटच्या विचार प्रक्रियेवर भर नाही — तर त्याच्या सभोवतालचे सर्व काही जे एजंटला प्रमाणात सुरक्षितपणे चालवण्यास सक्षम बनवते:

१. **टूल कॉल करणे** — ऑर्डर तपासणी आणि तिकीट तयार करणे.
२. **RAG** — ज्ञान आधारातून धोरणानुसार उत्तरं.
३. **स्मरणशक्ती** — संभाषणांदरम्यान ग्राहकाची आठवण ठेवणे.
४. **मॉडेल मार्गदर्शन** — सोप्या विनंत्यांना लहान मॉडेलकडे, क्लिष्टांना मोठ्या मॉडेलकडे पाठवणे.
५. **प्रतिसाद कॅशिंग** — पुन्हा विचारलेल्या प्रश्नांना मॉडेल कॉल न करता सेवा देणे.
६. **माणूस मंजुरी** — एक ठराविक मर्यादेपेक्षा जास्त परताव्यासाठी थांबाविणे.
७. **मूल्यमापन गेट** — खराब प्रकाशन रोखण्यासाठी ऑफलाइन चाचणी संच.
८. **निरीक्षणक्षमता** — प्रत्येक विनंतीभोवती OpenTelemetry ट्रेसिंग.

प्रत्येक विभाग स्वयंपूर्ण आणि चालवण्याजोगा आहे. प्रत्येक ओळ वाचा — उत्पादनासाठीच्या मूळ गोष्टी लक्षपूर्वक लहान ठेवले आहेत.


## सेटअप

या नोटबुक चालवण्यापूर्वी, खात्री करा की तुमच्याकडे आहे:

1. **एक Microsoft Foundry प्रोजेक्ट** ज्यामध्ये एक डिप्लॉय केलेला चॅट मॉडेल आहे (उदा. `gpt-5-mini`).
2. **Azure CLI सह लॉग इन केलेले** — तुमच्या टर्मिनलमध्ये `az login` चालवा.
3. **आवश्यक पर्यावरण चल सेट केलेले:**
   - `AZURE_AI_PROJECT_ENDPOINT` — तुमचा Microsoft Foundry प्रोजेक्ट endpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — तुमच्या डिप्लॉय केलेल्या मॉडेलचे नाव.

RAG विभाग वापरतो **Azure AI Search** जेव्हा `AZURE_SEARCH_SERVICE_ENDPOINT` आणि `AZURE_SEARCH_API_KEY` सेट केलेले असतात, आणि हे नसेल तर इन-मेमरी सर्चकडे fallback होते ज्यामुळे नोटबुक सर्च रीसोर्सशिवाय चालू शकतो.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. साधने

उत्पादन साधने खऱ्या प्रणालींच्या विरोधात खऱ्या काम करतात. येथे आपण साध्या Python फंक्शन्ससह ऑर्डर डेटाबेस आणि तिकीट प्रणालीचे अनुकरण करतो. `@tool` डेकोरेटर त्यांना एजंटला उपलब्ध करतो.

लक्षात घ्या की `issue_refund` उच्च मर्यादेपेक्षा जास्त परतफेडीसाठी `approval_mode="always_require"` वापरतो — हाच मानव-इन-द-लूप प्रिमिटिव आहे जो आपण नंतर तैनात करतो.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — धोरण ज्ञान आधार

धोरणाचे प्रश्न ("तुमचे परतावा विंडो काय आहे?") अधिकृत स्त्रोताकडून उत्तर दिले पाहिजे, मॉडेलच्या मेमरीकडून नाही. आम्ही एका लहान ज्ञान आधाराला शोध साधन म्हणून वापरतो.

उत्पादनात हे **Azure AI Search** आहे; येथे आम्ही इन-मेमरी कीवर्ड शोध पुरवतो जेणेकरून नोटबुक कुठेही चालू शकते, आणि वातावरणातील बदल सहज उपलब्ध असताना आपोआप Azure AI Search कडे स्विच होते.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. मेमरी

एक समर्थन एजंट जो कोणाशी बोलत आहे हे विसरतो तो एक वाईट समर्थन एजंट आहे. आम्ही प्रती कस्टमर एक लहान प्रोफाइल स्टोअर ठेवतो आणि एजंटच्या सूचना मध्ये एक छोटा सारांश टाकतो. उत्पादनात ही एक मेमरी सेवा आहे (पाठ 13 पहा); येथे एक dict पॅटर्न दृश्यमान करतो.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. मॉडेल राउटिंग आणि प्रतिसाद कॅशिंग

दोन खर्च लीवर्स एका सिंगल विनंती हाताळणाऱ्यासोबत जोडलेले:

- **राउटिंग**: एक स्वस्त हीयुरिस्टिक वर्गीकारक ठरवते की विनंतीला लहान मॉडेल आवश्यक आहे का मोठ्या मॉडेलची.
- **कॅशिंग**: सामान्यीकृत पुनरावृत्ती प्रश्न कॅशमधून थेट दिले जातात ज्यात मॉडेल कॉल नाही.

वर्गीकारक इथे जानबूजून सोपा आहे. उत्पादनात आपण त्याला ट्रॅफिकच्या विरुद्ध प्रमाणित करू शकता आणि तो Foundry च्या मॉडेल राउटरने बदलू शकता.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-5-nano
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-5-mini

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. एजंट, मानवी मंजुरी, आणि निरीक्षण

आता आपण वरील साधनांमधून एजंट तयार करतो आणि प्रत्येक विनंतीला OpenTelemetry स्पॅनमध्ये संलग्न करतो. `handle_support_request` फंक्शन हे उत्पादन विनंती हाताळणारे आहे: कॅशे → मार्गदर्शन → ट्रेस → चालवा → कॅशे.

मानवी मंजुरी फ्रेमवर्कद्वारे हाताळली जाते: कारण `issue_refund` ला `approval_mode="always_require"` सेट केले आहे, रन थांबतो आणि आम्ही स्पष्टपणे निराकरण करतो अशी मंजुरीची विनंती दिसते.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

# Build one agent per model tier so we can route by cost. The current agent-framework
# selects the model on the client, so each tier gets its own FoundryChatClient.
_TOOLS = [get_order_status, open_ticket, search_policies, issue_refund]
_agents_by_model: dict[str, object] = {}


def agent_for(model_name: str):
    if model_name not in _agents_by_model:
        client = FoundryChatClient(
            project_endpoint=endpoint,
            model=model_name,
            credential=AzureCliCredential(),
        )
        _agents_by_model[model_name] = client.as_agent(
            name="ContosoSupportAgent",
            instructions=SUPPORT_INSTRUCTIONS,
            tools=_TOOLS,
        )
    return _agents_by_model[model_name]


# Default agent (used by the evaluation gate, which does not route).
support_agent = agent_for(SMALL_MODEL)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await agent_for(chosen_model).run(prompt)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. मूल्यमापन दुवा

हा धडा पासूनच्या रिलीज गेट आहे: एक ऑफलाइन टेस्ट सेट एजंटचे स्कोअर करते, आणि तैनाती पुढे सुरू होते फक्त जेव्हा पास दर एखाद्या थ्रेशोल्डवरून पुढे जाते. येथील स्कोरर अगदी सोपा कीवर्ड-ओवरलॅप तपासणी आहे ज्यामुळे नोटबुक स्व-संबंधीत राहतो; उत्पादनात आपण LLM-एज-जज किंवा फ्रेमवर्क मूल्यमापक वापराल (धडा १० पहा).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## एकत्र ठेवणे: एक अनुकरणात्मक प्रकाशन

खालील सेल संपूर्ण लूप दाखवतो ज्याचे वर्णन धड्यात केले आहे: मूल्यमापन द्वार चालवा, आणि जर ते उत्तीर्ण झाले तरच "प्रकाशित" करा. हेच नमुना आपण CI मध्ये फाउंडरी एजंट सेवा कडे एजंट आवृत्ती प्रोत्साहित करण्यापूर्वी चालवाल.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## सारांश

आपण प्रत्येक ऑपरेशनल बाबींचा समावेश असलेला उत्पादनास तयार ग्राहक समर्थन एजंट तयार केला आहे:

- **टूल्स, RAG, आणि मेमरी** एजंटला क्षमता आणि संदर्भ देतात.
- **मॉडेल रूटिंग आणि कॅशिंग** लेटेंसी आणि खर्च नियंत्रणात ठेवतात.
- **मानवी मान्यता** मोठ्या परताव्यासारखे उच्च-धोकादायक क्रिया संरक्षित करतात.
- **मूल्यांकन गेट** खराब प्रकाशने प्रक्षिप्त होण्याआधी थांबवते.
- **ट्रेसिंग** प्रत्येक विनंतीचे निरीक्षण करण्यायोग्य बनवते.

### आव्हान

हा एजंट विस्तृत करा:

1. **अनेक मॉडेलांना समर्थन द्या** — तिसऱ्या "तर्कशास्त्र" स्तराची भर घाला आणि तक्रारी/विषय व्यवस्थापन त्याकडे मार्गदर्शित करा.
2. **मूल्यांकन गेट जोडा** — `TEST_CASES` मध्ये परतावा-मान्यता परिस्थिती समाविष्ट करा आणि गेटने मागे पडण्यांना अडवले आहे का याची खात्री करा.
3. **खर्च-जाणणारी मार्गदर्शना जोडा** — प्रत्येक विनंतीचा अंदाजित खर्च (लहान विरुद्ध मोठा विरुद्ध कॅशे) ट्रॅक करा आणि मिसळलेल्या प्रश्नांच्या बॅचनंतर खर्च अहवाल छापा.

पुढील धड्यात आपण विरुद्ध प्रवास करून Microsoft Foundry Local आणि Qwen सह पूर्णपणे आपल्या स्वतःच्या संगणकावर एजंट चालवणार आहात.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
